# 노션 트러블슈팅 검색기


**프로젝트 개요**

기존 노션 검색의 한계인 제목 기반 검색을 벗어나, 문서 내용까지 의미적으로 검색할 수 있는 시스템을 구현했습니다.
```

노션 DB Export (Markdown)
        ↓
문서 청킹 (RecursiveCharacterTextSplitter)
        ↓
임베딩 변환 (OpenAI Embeddings)
        ↓
벡터 DB 저장 (Chroma)
        ↓
의미 기반 검색
        ↓
관련 문서 반환
```


## 노션에서 Export 한 zip 파일 압축 해제

In [12]:
import zipfile
import os

# 압축 풀기
with zipfile.ZipFile("data.zip", "r") as zip_ref:
    zip_ref.extractall("./notion_docs")

print("압축 해제 완료!")

# 파일 목록 확인
for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        print(file)

압축 해제 완료!
.DS_Store
트러블슈팅 ae9e1903be104960aed1364937c6d431_all.csv
트러블슈팅 ae9e1903be104960aed1364937c6d431.csv
ExportBlock-f89a79cd-0b01-4d26-abdc-e7b047031d39-Part-1.zip
PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180debcdcd3e399ada17e.md
@[트러블슈팅]MySQL 버전 문제 (ONLY_FULL_GROUP_BY 모드) 292aca7eedd180ef858bc549f96f0b7f.md
네트워크 699e097547da496d9b5e57952d80156d.md
springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
db 이관 후 테이블 대소문자 구분 안됨 ae65f04015184caaa30013127611c1c6.md
queryDSL - collate = syntaxError 7f33a2a0a4d74ad7916310f3264c4e6b.md
윈도우 jenkins - 젠킨스 스크립트가 종료되면 jar 도 종료됨 “dontkillMe adbcd0241c88414e98bc24890ad267e5.md
Data truncated for column 'orders_item_pid' at row 17daca7eedd18084b009ead9923292f2.md
@Vaild 유효성체크 안될때 0e803aa494fc45c1a0be0e43d17b8bfe.md
동일컬럼에 대해 객체와 필드로 매핑했을 때 - Column is 

In [13]:
# 전체 파일 구조 확인 먼저!
import os

for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            print(full_path)

./notion_docs/troubleshooting/PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180debcdcd3e399ada17e.md
./notion_docs/troubleshooting/@[트러블슈팅]MySQL 버전 문제 (ONLY_FULL_GROUP_BY 모드) 292aca7eedd180ef858bc549f96f0b7f.md
./notion_docs/troubleshooting/네트워크 699e097547da496d9b5e57952d80156d.md
./notion_docs/troubleshooting/springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
./notion_docs/troubleshooting/db 이관 후 테이블 대소문자 구분 안됨 ae65f04015184caaa30013127611c1c6.md
./notion_docs/troubleshooting/queryDSL - collate = syntaxError 7f33a2a0a4d74ad7916310f3264c4e6b.md
./notion_docs/troubleshooting/윈도우 jenkins - 젠킨스 스크립트가 종료되면 jar 도 종료됨 “dontkillMe adbcd0241c88414e98bc24890ad267e5.md
./notion_docs/troubleshooting/Data truncated for column 'orders_item_pid' at row 17daca7eedd18084b009ead9923292f2.md
./notion_docs/troubleshooting/@Vaild 유효성체크 안될때 0e803aa494fc4

In [14]:
!pip install -qU langchain-chroma
!pip install -qU langchain-google-genai
!pip install -qU langchain-openai
!pip install langchain-text-splitters -q


zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


### API 키 설정

In [28]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import json

load_dotenv()
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

# md 파일 전부 읽기
docs = []
file_names = []

for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            with open(full_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():
                    docs.append(content)
                    file_names.append(file)

print(f"📄 총 {len(docs)}개 문서 로드!")
print("\n파일 목록:")
for name in file_names[:30]:
    print(f"  - {name}")


/Users/eunchae/repository/skax


True

In [29]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 변경!
import os


# md 파일 전부 읽기
docs = []
file_names = []

for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            with open(full_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():  # 빈 파일 제외
                    docs.append(content)
                    file_names.append(file)

print(f"📄 총 {len(docs)}개 문서 로드!")
print("\n파일 목록:")
for name in file_names:
    print(f"  - {name}")

📄 총 56개 문서 로드!

파일 목록:
  - PR 서버에 적용하면서 문제 및 개선사항 정리 (feat dentrion_tablet_ap 19aaca7eedd180debcdcd3e399ada17e.md
  - @[트러블슈팅]MySQL 버전 문제 (ONLY_FULL_GROUP_BY 모드) 292aca7eedd180ef858bc549f96f0b7f.md
  - 네트워크 699e097547da496d9b5e57952d80156d.md
  - springdoc - array 타입이 스웨거에서 이상하게 표시 198aca7eedd18065b211d0967839d244.md
  - db 이관 후 테이블 대소문자 구분 안됨 ae65f04015184caaa30013127611c1c6.md
  - queryDSL - collate = syntaxError 7f33a2a0a4d74ad7916310f3264c4e6b.md
  - 윈도우 jenkins - 젠킨스 스크립트가 종료되면 jar 도 종료됨 “dontkillMe adbcd0241c88414e98bc24890ad267e5.md
  - Data truncated for column 'orders_item_pid' at row 17daca7eedd18084b009ead9923292f2.md
  - @Vaild 유효성체크 안될때 0e803aa494fc45c1a0be0e43d17b8bfe.md
  - 동일컬럼에 대해 객체와 필드로 매핑했을 때 - Column is duplicated in  17daca7eedd180d8b173fd87460dfc49.md
  - tablet 모듈 신규 생성 시 bean 주입 에러 e14a4b3c072c4cf48f05f

In [30]:
from langchain_core.documents import Document


# 1. md 파일 읽기 (파일명 포함!)
docs = []
for root, dirs, files in os.walk("./notion_docs"):
    for file in files:
        if file.endswith(".md"):
            full_path = os.path.join(root, file)
            with open(full_path, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():
                    docs.append(Document(
                        page_content=content,
                        metadata={"source": file}  # 파일명 저장!
                    ))

In [ ]:
# 4. 질의 정규화 및 동의어 확장 검색
query_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

query_prompt = ChatPromptTemplate.from_template("""
너는 트러블슈팅 검색을 위한 질의 정규화기다.

목표:
- 사용자 질문에서 핵심 기술 키워드를 추출
- 한글/영문/대소문자가 다른 동일 개념을 같은 의미로 처리
- 검색에 유리하도록 대표 검색어와 확장 검색어를 생성

동의어 규칙 예시:
- jenkins = Jenkins = 젠킨스
- github = GitHub = 깃허브
- swagger = Swagger = 스웨거
- spring boot = Spring Boot = 스프링부트
- mysql = MySQL = 마이에스큐엘
- tls = TLS = ssl = SSL

지침:
- 같은 의미의 표현은 하나의 개념으로 묶기
- 원문 의미를 바꾸지 말기
- 검색에 불필요한 조사나 장식어는 제거 가능
- 출력은 반드시 JSON 형태로만 반환

출력 형식:
{{
  "normalized_query": "대표 검색어",
  "expanded_keywords": ["동의어1", "동의어2", "동의어3"],
  "intent": "사용자가 찾고 싶은 문제 상황 요약"
}}

사용자 질문:
{question}
""")

def normalize_query(question: str):
    response = query_llm.invoke(query_prompt.format(question=question))
    return json.loads(response.content)

def search_documents(vector_store, question: str, k: int = 5):
    query_info = normalize_query(question)

    normalized_query = query_info["normalized_query"]
    expanded_keywords = query_info["expanded_keywords"]

    queries = [normalized_query] + expanded_keywords

    results = []
    seen = set()

    for q in queries:
        docs = vector_store.similarity_search(q, k=k)
        for doc in docs:
            key = (doc.metadata.get("source", ""), doc.page_content[:200])
            if key not in seen:
                seen.add(key)
                results.append(doc)

    return {
        "query_info": query_info,
        "documents": results[:k]
    }


In [ ]:
question = "젠킨스 배포할 때 빌드가 실패해요"

result = search_documents(vector_store, question, k=5)

print("정규화 결과:")
print(json.dumps(result["query_info"], ensure_ascii=False, indent=2))
print()
print(f"검색 결과 {len(result['documents'])}개\n")

for i, doc in enumerate(result["documents"], 1):
    print(f"=== {i}번 문서 ===")
    print(f"📄 제목: {doc.metadata.get('source', 'unknown')}")
    print(f"📝 내용: {doc.page_content[:300]}")
    print()
